# DS317 - TRÍCH XUẤT BỘ DỮ LIỆU MẪU DEMO (39 THUỘC TÍNH: PREP27 + RTMA12)

**Thành viên phụ trách:** Quách Tuấn Kiệt (MSSV: 24520911) - Kỹ sư Dữ liệu mẫu & Chất lượng DAMA

**Mục tiêu (Bám sát `Kế hoạch hoàn thiện và phân công công việc cuối kỳ (1).md` & `04_Data/README.md`):**
1. Trích xuất bộ dữ liệu mẫu chuẩn **`3,000` dòng** (nằm trong dải `1,000 - 5,000` dòng) từ tập kiểm thử đã tiền xử lý (`03_ban_tien_xu_ly_27_thuoc_tinh/q4.parquet` hoặc `test.parquet`).
2. **Phân tầng đại diện đủ 3 nhóm chân trời thời gian** trong mốc tác nghiệp `<= 168h`:
   - Nhóm 1 (`< 24h`): `1,000` dòng ($0 < \text{Target\_Hours} < 24$)
   - Nhóm 2 (`24h - 72h`): `1,000` dòng ($24 \le \text{Target\_Hours} < 72$)
   - Nhóm 3 (`72h - 168h`): `1,000` dòng ($72 \le \text{Target\_Hours} \le 168$)
3. **Ghép nối chuẩn xác 12 cột thời tiết NOAA RTMA 2025** từ `weather_rtma_2025_sidecar` theo logic đối chiếu khóa `(mmsi, trip_id, base_date_time)` và kiểm tra chống rò rỉ tương lai (`rtma_published_at <= base_date_time`) + QC NDBC (`DPT <= TMP`) tham khảo từ `pipeline_add_weather.ipynb`.
4. Xuất ra `demo_sample_3000.csv` và `demo_sample_3000.parquet` (nén `zstd`, đủ **39 cột**) để thay thế bản placeholder trong `04_Data/` phục vụ Web Demo Streamlit (`app.py`).

> **Hướng dẫn gắn Dataset trên Kaggle trước khi Run All:**
> 1. `kienvo134/noaa-ais-2025-full-year-tin-x-l-nn-tng` (chứa `03_ban_tien_xu_ly_27_thuoc_tinh/`)
> 2. `kienvo134/dataset-noaa-rtma-2025` (chứa `weather_rtma_2025_sidecar/`)

In [ ]:
"""Cell 1: Khai báo thư viện, cấu hình tham số và Schema 39 thuộc tính chuẩn."""
import gc
import glob
import os
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

IS_KAGGLE = os.path.exists('/kaggle/input')
OUT_DIR = '/kaggle/working' if IS_KAGGLE else './demo_extract_out'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_PER_HORIZON = 1000  # 3 nhóm x 1,000 = 3,000 dòng (chuẩn 1,000 - 5,000 dòng)
BATCH_SIZE = 1_000_000
CANDIDATE_POOL_PER_BUCKET = 50_000  # Số lượng mẫu ứng viên tối đa mỗi nhóm để tiết kiệm RAM

AIS_SUBDIR = '03_ban_tien_xu_ly_27_thuoc_tinh'
SIDECAR_SUBDIRS = ['weather_rtma_2025_sidecar', 'weather_rtma_2025_sidecar_leakfree']
KEYS = ['mmsi', 'trip_id', 'base_date_time']
TIME_COLS = ['rtma_analysis_time', 'rtma_published_at']

# 27 cột chuẩn tiền xử lý (theo 04_Data/README.md)
SCHEMA_27_COLS = [
    'mmsi', 'trip_id', 'base_date_time', 'longitude', 'latitude', 'sog', 'cog', 'heading',
    'cog_sin', 'cog_cos', 'heading_sin', 'heading_cos', 'status', 'vessel_type',
    'length', 'width', 'draft', 'cargo', 'berth_lon', 'berth_lat', 'ata_time',
    'Dist_To_Berth_NM', 'Area_Proxy', 'Volume_Proxy', 'Is_Tanker', 'imputation_flag', 'Target_Hours'
]

# 12 cột thời tiết NOAA RTMA 2025 (theo 04_Data/README.md & pipeline_add_weather.ipynb)
WEATHER_10_CONT = [
    'rtma_TMP_C', 'rtma_DPT_C', 'rtma_PRES_hpa', 'rtma_VIS_nm', 'rtma_WIND_knots',
    'rtma_GUST_knots', 'rtma_wdir_sin', 'rtma_wdir_cos', 'rtma_headwind_knots', 'rtma_crosswind_knots'
]
WEATHER_12_COLS = ['rtma_domain', 'weather_missing_flag'] + WEATHER_10_CONT
ALL_39_COLS = SCHEMA_27_COLS + WEATHER_12_COLS

# Giới hạn vật lý NDBC 6.2.8 / Bảng 6 (tham chiếu pipeline_add_weather.ipynb)
NDBC_LIMITS = {
    'rtma_TMP_C': (-50.0, 55.0),
    'rtma_DPT_C': (-55.0, 45.0),
    'rtma_PRES_hpa': (850.0, 1080.0),
    'rtma_VIS_nm': (0.0, 50.0),
    'rtma_WIND_knots': (0.0, 120.0),
    'rtma_GUST_knots': (0.0, 150.0),
}
print(f'[OK] Cấu hình sẵn sàng | OUT_DIR={OUT_DIR} | Tổng số cột mục tiêu={len(ALL_39_COLS)}')

In [ ]:
"""Cell 2: Tự động dò tìm tệp AIS 27 thuộc tính (Q4/Test) và Sidecar thời tiết RTMA."""

def _find_first(patterns, check=None):
    for pat in patterns:
        for path in sorted(glob.glob(pat, recursive=True)):
            if check is None or check(path):
                return path
    return None

def _is_ais_27(path):
    names = pq.read_schema(path).names
    return len(names) == 27 and 'cog_sin' in names and 'imputation_flag' in names and 'rtma_domain' not in names

def locate_demo_inputs():
    # Ưu tiên q4.parquet theo chỉ định tại 04_Data/README.md, fallback sang test.parquet
    ais_q4 = _find_first([f'/kaggle/input/**/{AIS_SUBDIR}/q4.parquet', f'./**/{AIS_SUBDIR}/q4.parquet'], _is_ais_27)
    ais_test = _find_first([f'/kaggle/input/**/{AIS_SUBDIR}/test.parquet', f'./**/{AIS_SUBDIR}/test.parquet'], _is_ais_27)
    ais_path = ais_q4 or ais_test
    if not ais_path:
        raise FileNotFoundError(f'Không tìm thấy q4.parquet hoặc test.parquet trong {AIS_SUBDIR}. Hãy gắn dataset kienvo134/noaa-ais-2025-full-year-tin-x-l-nn-tng!')

    # Quét tất cả file parquet của sidecar thời tiết (test.parquet / test_part*.parquet)
    sidecar_files = []
    for sub in SIDECAR_SUBDIRS:
        found = sorted(glob.glob(f'/kaggle/input/**/{sub}/test*.parquet', recursive=True))
        if not found:
            found = sorted(glob.glob(f'./**/{sub}/test*.parquet', recursive=True))
        if found:
            sidecar_files = found
            break
    if not sidecar_files:
        # Thử tìm tất cả parquet trong sidecar nếu không đặt tên test*
        for sub in SIDECAR_SUBDIRS:
            found = sorted(glob.glob(f'/kaggle/input/**/{sub}/*.parquet', recursive=True))
            if found:
                sidecar_files = found
                break
    if not sidecar_files:
        raise FileNotFoundError('Không tìm thấy sidecar thời tiết RTMA. Hãy gắn dataset kienvo134/dataset-noaa-rtma-2025!')

    return ais_path, sidecar_files

AIS_PATH, SIDECAR_FILES = locate_demo_inputs()
SIDE_SCHEMA = pq.read_schema(SIDECAR_FILES[0]).names
SIDE_HAS_TIME = all(c in SIDE_SCHEMA for c in TIME_COLS)
print(f'[AIS Source] {AIS_PATH} (rows={pq.ParquetFile(AIS_PATH).metadata.num_rows:,})')
print(f'[Sidecar]    {len(SIDECAR_FILES)} file(s), ví dụ: {os.path.basename(SIDECAR_FILES[0])} | SIDE_HAS_TIME={SIDE_HAS_TIME}')

In [ ]:
"""Cell 3: Quét phân tầng & trích xuất 3,000 dòng đại diện đủ 3 nhóm chân trời (<= 168h)."""
t0 = time.time()
rng = np.random.default_rng(RANDOM_STATE)

# Reservoir / capped pool cho 3 nhóm chân trời theo yêu cầu kế hoạch:
# B1: < 24h | B2: 24h - 72h | B3: 72h - 168h
pools = {'<24h': [], '24h-72h': [], '72h-168h': []}
pool_counts = {k: 0 for k in pools}

pf_ais = pq.ParquetFile(AIS_PATH)
for batch in pf_ais.iter_batches(batch_size=BATCH_SIZE, columns=SCHEMA_27_COLS):
    df_b = batch.to_pandas()
    # Điều kiện lọc sạch cho Demo: hành trình đang chạy (1.0 <= sog <= 40.0), loại neo/buộc bến, có tọa độ bến hợp lệ
    mask_clean = (
        (df_b['Target_Hours'] > 0.0) & (df_b['Target_Hours'] <= 168.0) &
        (df_b['sog'] >= 1.0) & (df_b['sog'] <= 40.0) &
        (~df_b['status'].isin([1.0, 5.0])) &
        df_b['berth_lon'].notna() & df_b['berth_lat'].notna() &
        df_b['Dist_To_Berth_NM'].notna()
    )
    df_b = df_b.loc[mask_clean]
    if len(df_b) == 0:
        continue

    buckets = {
        '<24h': df_b[df_b['Target_Hours'] < 24.0],
        '24h-72h': df_b[(df_b['Target_Hours'] >= 24.0) & (df_b['Target_Hours'] < 72.0)],
        '72h-168h': df_b[(df_b['Target_Hours'] >= 72.0) & (df_b['Target_Hours'] <= 168.0)],
    }
    for b_name, sub_df in buckets.items():
        if len(sub_df) > 0 and pool_counts[b_name] < CANDIDATE_POOL_PER_BUCKET:
            take_n = min(len(sub_df), max(500, CANDIDATE_POOL_PER_BUCKET // 10))
            sampled = sub_df.sample(n=take_n, random_state=RANDOM_STATE) if len(sub_df) > take_n else sub_df
            pools[b_name].append(sampled)
            pool_counts[b_name] += len(sampled)

    if all(v >= CANDIDATE_POOL_PER_BUCKET for v in pool_counts.values()):
        break

selected_parts = []
for b_name, parts in pools.items():
    df_pool = pd.concat(parts, ignore_index=True).drop_duplicates(subset=KEYS)
    assert len(df_pool) >= N_PER_HORIZON, f'Nhóm {b_name} chỉ có {len(df_pool)} dòng sạch, cần >= {N_PER_HORIZON}'
    # Lấy mẫu cân bằng theo Is_Tanker (nếu đủ) để đại diện cả tàu hàng & tàu dầu
    df_sampled = df_pool.sample(n=N_PER_HORIZON, random_state=RANDOM_STATE).copy()
    df_sampled['_horizon_bucket'] = b_name
    selected_parts.append(df_sampled)
    print(f'  + Nhóm {b_name:8s}: Pool={len(df_pool):,} -> Chọn={len(df_sampled):,} dòng | Tanker={(df_sampled["Is_Tanker"]==1).mean()*100:.1f}%')

df_ais_demo = pd.concat(selected_parts, ignore_index=True)
del pools, selected_parts
gc.collect()
print(f'[OK] Đã chọn {len(df_ais_demo):,} dòng AIS sạch trong {time.time()-t0:.1f}s')

In [ ]:
"""Cell 4: Ghép nối thời tiết RTMA theo khóa (mmsi, trip_id, base_date_time) & QC NDBC."""
t1 = time.time()
target_trips = set(df_ais_demo['trip_id'].astype(str).unique())
side_cols_to_read = list(dict.fromkeys(KEYS + WEATHER_12_COLS + (TIME_COLS if SIDE_HAS_TIME else [])))

sidecar_matches = []
for s_file in SIDECAR_FILES:
    pf_s = pq.ParquetFile(s_file)
    for batch in pf_s.iter_batches(batch_size=BATCH_SIZE, columns=side_cols_to_read):
        # Lọc nhanh theo trip_id trước khi chuyển sang pandas đầy đủ
        df_s = batch.to_pandas()
        mask_t = df_s['trip_id'].astype(str).isin(target_trips)
        if mask_t.any():
            sidecar_matches.append(df_s.loc[mask_t])

if not sidecar_matches:
    raise RuntimeError('Không tìm thấy bản ghi khớp trip_id trong sidecar thời tiết! Kiểm tra lại q4.parquet vs test.parquet.')

df_side_sub = pd.concat(sidecar_matches, ignore_index=True)
# Chuẩn hóa kiểu khóa trước khi merge để bảo đảm khớp chính xác 100%
for df_tmp in (df_ais_demo, df_side_sub):
    df_tmp['_k_mmsi'] = pd.to_numeric(df_tmp['mmsi'], errors='coerce').astype('int64')
    df_tmp['_k_trip'] = df_tmp['trip_id'].astype(str)
    df_tmp['_k_bdt'] = pd.to_datetime(df_tmp['base_date_time']).astype('datetime64[us]')

df_side_sub = df_side_sub.drop_duplicates(subset=['_k_mmsi', '_k_trip', '_k_bdt'])

# Merge 1-1 an toàn theo khóa
merge_cols = ['_k_mmsi', '_k_trip', '_k_bdt'] + WEATHER_12_COLS + (TIME_COLS if SIDE_HAS_TIME else [])
df_merged = df_ais_demo.merge(
    df_side_sub[merge_cols],
    on=['_k_mmsi', '_k_trip', '_k_bdt'],
    how='left',
    validate='one_to_one'
)

# Kiểm tra chống rò rỉ thời gian phát hành RTMA (tham chiếu pipeline_add_weather.ipynb Cell 3)
if SIDE_HAS_TIME and 'rtma_published_at' in df_merged.columns:
    pub_dt = pd.to_datetime(df_merged['rtma_published_at'])
    bdt_dt = pd.to_datetime(df_merged['base_date_time'])
    n_leak = int((pub_dt > bdt_dt).sum())
    assert n_leak == 0, f'PHÁT HIỆN RÒ RỈ TƯƠNG LAI: {n_leak} dòng có rtma_published_at > base_date_time!'
    print('[QC Anti-Leakage] OK: 0 dòng dùng RTMA phát hành sau thời điểm bản tin.')

# QC vật lý khí tượng NDBC 6.2.8 / Bảng 6 (tham chiếu pipeline_add_weather.ipynb Cell 4)
if 'rtma_DPT_C' in df_merged.columns and 'rtma_TMP_C' in df_merged.columns:
    m_dpt = df_merged['rtma_DPT_C'].notna() & df_merged['rtma_TMP_C'].notna() & (df_merged['rtma_DPT_C'] > df_merged['rtma_TMP_C'])
    if m_dpt.sum() > 0:
        df_merged.loc[m_dpt, 'rtma_DPT_C'] = df_merged.loc[m_dpt, 'rtma_TMP_C']
        print(f'[QC NDBC 6.2.8] Đã hiệu chỉnh {int(m_dpt.sum())} dòng DPT > TMP -> DPT = TMP')

for col_lim, (lo, hi) in NDBC_LIMITS.items():
    if col_lim in df_merged.columns:
        m_out = df_merged[col_lim].notna() & ((df_merged[col_lim] < lo) | (df_merged[col_lim] > hi))
        if m_out.sum() > 0:
            df_merged.loc[m_out, col_lim] = np.nan
            print(f'[QC NDBC Bảng 6] Đã gán NaN cho {int(m_out.sum())} giá trị ngoài miền ở {col_lim}')

# Xáo trộn lại 3,000 dòng và sắp xếp đúng 39 cột chuẩn
df_demo = df_merged[ALL_39_COLS].sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'[OK] Ghép thời tiết hoàn tất trong {time.time()-t1:.1f}s | Shape={df_demo.shape}')

In [ ]:
"""Cell 5: Ép kiểu chuẩn PyArrow, kiểm định chất lượng và xuất CSV + Parquet."""
# Chuẩn hóa đúng kiểu dữ liệu theo 04_Data/README.md
int64_cols = ['mmsi']
int32_cols = ['vessel_type', 'Is_Tanker', 'imputation_flag']
int8_cols = ['rtma_domain', 'weather_missing_flag']
str_cols = ['trip_id']
dt_cols = ['base_date_time', 'ata_time']

for c in int64_cols:
    df_demo[c] = df_demo[c].astype('int64')
for c in int32_cols:
    df_demo[c] = df_demo[c].astype('int32')
for c in int8_cols:
    df_demo[c] = df_demo[c].fillna(0 if c == 'rtma_domain' else 1).astype('int8')
for c in str_cols:
    df_demo[c] = df_demo[c].astype(str)
for c in dt_cols:
    df_demo[c] = pd.to_datetime(df_demo[c]).astype('datetime64[us]')

float32_cols = [c for c in ALL_39_COLS if c not in (int64_cols + int32_cols + int8_cols + str_cols + dt_cols)]
for c in float32_cols:
    df_demo[c] = df_demo[c].astype('float32')

# Kiểm định cuối trước khi xuất
assert list(df_demo.columns) == ALL_39_COLS, 'Sai thứ tự hoặc thiếu cột trong 39 thuộc tính!'
assert len(df_demo) == 3 * N_PER_HORIZON, f'Số dòng không khớp {3 * N_PER_HORIZON}'

n_b1 = int((df_demo['Target_Hours'] < 24.0).sum())
n_b2 = int(((df_demo['Target_Hours'] >= 24.0) & (df_demo['Target_Hours'] < 72.0)).sum())
n_b3 = int(((df_demo['Target_Hours'] >= 72.0) & (df_demo['Target_Hours'] <= 168.0)).sum())
wx_ok_pct = float((df_demo['weather_missing_flag'] == 0).mean() * 100)
tanker_pct = float((df_demo['Is_Tanker'] == 1).mean() * 100)

csv_out = os.path.join(OUT_DIR, 'demo_sample_3000.csv')
pq_out = os.path.join(OUT_DIR, 'demo_sample_3000.parquet')
df_demo.to_csv(csv_out, index=False)
df_demo.to_parquet(pq_out, index=False, compression='zstd')

print('===================================================================')
print('  BÁO CÁO KIỂM ĐỊNH BỘ DỮ LIỆU MẪU DEMO (39 THUỘC TÍNH)')
print('===================================================================')
print(f'  - Tổng số dòng / cột : {len(df_demo):,} dòng x {df_demo.shape[1]} cột')
print(f'  - Số tàu độc nhất    : {df_demo["mmsi"].nunique():,} mmsi | {df_demo["trip_id"].nunique():,} trips')
print(f'  - Phân bổ chân trời  : <24h={n_b1} | 24-72h={n_b2} | 72-168h={n_b3}')
print(f'  - Tỷ lệ Tanker       : {tanker_pct:.2f}%')
print(f'  - Tỷ lệ khớp RTMA    : {wx_ok_pct:.2f}% (weather_missing_flag == 0)')
print(f'  - Tọa độ bến hợp lệ  : {df_demo["berth_lon"].notna().mean()*100:.2f}%')
print(f'  - Xuất CSV           : {csv_out} ({os.path.getsize(csv_out):,} bytes)')
print(f'  - Xuất Parquet (zstd): {pq_out} ({os.path.getsize(pq_out):,} bytes)')
print('===================================================================')